In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
import gc
import skimage
import joblib
import seaborn as sns
from pathlib import Path 
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from skimage import io, transform, color, filters, measure, feature, morphology
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.color import rgb2gray
from skimage.segmentation import clear_border
from skimage.io import imread
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter
from joblib import Parallel, delayed
from tqdm.auto import tqdm

### Dataframe
- each row has corresponding MSI and HSI paths (1:1 image correspondence) for raw and preprocessed images
- load in saved CSV file

In [ ]:
df = pd.read_csv("kidney_metadata_final.csv") # reading from saved CSV
# print(df.head()) # for verification

### Global Declarations and Parameters
- class KidneyDataset: for CNN (separate HSI/MSI)
- class DualKidneyDataset: for CNN (late fusion)
- cnn_separate_metrics: classification report + confusion matrix for CNN (separate HSI/MSI)
- cnn_fusion_metrics: classification report + confusion matrix for CNN (late fusion)

In [ ]:
img_size = (128, 128)
batch = 16
seed = 8 # for reproducibility
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# CNN (separate HSI/MSI models)
class KidneyDataset(Dataset):
    
    # extracts file paths (from dataframe, CSV file)
    def __init__(self, paths, labels, img_size=img_size):
        self.paths = paths
        self.labels = labels
        self.img_size = img_size
        
    # how many datapoints in dataset (to indicate end of epoch)
    def __len__(self):
        return len(self.paths)

    # converts Numpy array into PyTorch tensor
    def _process_image(self, path):
        image = cv2.imread(path)
        if image is None: return torch.zeros((3, self.img_size[0], self.img_size[1])) # safety net
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0 # normalize
        image = np.transpose(image, (2,0,1)) # (channels, height, width)
        return torch.from_numpy(image) 

    # retrieves desired image sample and corresponding label
    def __getitem__(self, idx):
        img = self._process_image(self.paths[idx])
        label = torch.tensor(self.labels[idx], dtype = torch.float32).unsqueeze(0)
        return img, label

In [ ]:
# CNN (late fusion: Y-head input)
class DualKidneyDataset(Dataset):

    # extracts file paths (from dataframe, CSV file)
    def __init__(self, paths_hsi, paths_msi, labels, img_size=(img_size[0], img_size[1])):
        self.paths_hsi = paths_hsi
        self.paths_msi = paths_msi
        self.labels = labels
        self.img_size = img_size

    # how many datapoints in dataset (to indicate end of epoch)
    def __len__(self):
        return len(self.paths_hsi) # same as self.paths_msi

    # converts Numpy array to PyTorch tensor
    def _process_image(self, path):
        image = cv2.imread(path)
        if image is None: return torch.zeros((3, self.img_size[0], self.img_size[1])) # safety net
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.img_size)
        image = image.astype(np.float32) / 255.0 # normalize
        image = np.transpose(image, (2,0,1)) # (channels, height, width)
        return torch.from_numpy(image)

    # retrieves desired image sample and corresponding label
    def __getitem__(self, idx):
        img_hsi = self._process_image(self.paths_hsi[idx])
        img_msi = self._process_image(self.paths_msi[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float32).unsqueeze(0)
        return img_hsi, img_msi, label

In [ ]:
# confusion matrices and classification reports

# CNN (separate HSI/MSI models)
def cnn_separate_metrics(model, loader, title):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad(): # saves memory
        for images, labels in loader: # one batch per for loop
            outputs = model(images.to(device))
            preds = (outputs > 0.5).float() # clamp prediction values to 0 and 1  
            all_preds.extend(preds.cpu().numpy()) # add batch results to list
            all_labels.extend(labels.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure() # plotting confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Tumor'], yticklabels=['Normal', 'Tumor'])
    plt.title(f'Confusion Matrix: {title}')
    plt.show()
    print(classification_report(all_labels, all_preds, target_names=['Normal', 'Tumor'], digits=6)) 

# CNN (late fusion)
def cnn_fusion_metrics(model, loader, title): 
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad(): # saves memory
        for img_hsi, img_msi, labels in loader: # one batch per for loop
            outputs = model(img_hsi.to(device), img_msi.to(device))
            preds = (outputs > 0.5).float() # clamp prediction values to 0 and 1
            all_preds.extend(preds.cpu().numpy()) # add batch results to list
            all_labels.extend(labels.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure() # plotting confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Tumor'], yticklabels=['Normal', 'Tumor'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix: {title}')
    plt.show()
    print(classification_report(all_labels, all_preds, target_names=['Normal', 'Tumor'], digits=6))

### CNN (separate HSI/MSI models)
- conv2D layers with 3x3 kernel (16 then 32 filters, ReLU activation)
- maxPooling2D with 2x2 kernel
- dropout layers (variable values for hyperparameter tuning)
- rearning rate (variable values for hyperparameter tuning)

In [ ]:
class CNN(nn.Module):
    
    # define all CNN layers
    def __init__(self, dropout_rate=0.3):
        super(CNN, self).__init__() # initialize parent nn.Module class
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1) # padding = 1 keeps image size as img_size after convolution
        self.bn1 = nn.BatchNorm2d(16) # zero mean and unit std for intermediate layers' activations (stabilization and regularization)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.drop = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(32*32*32, 32) # condense  32^3 input to a final 32 features
        self.fc2 = nn.Linear(32, 1) # condense 32 features into singular probabilistic number
    
    # define data flow through CNN layers in __init__ function
    def forward(self, x):
        if isinstance(x, dict):  x = x['input_hsi'] 
        # first layer: 2d conv -> batch normalization -> ReLU -> 2d pooling -> dropout
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.drop(x)
        # second layer: 2d conv -> batch normalization -> flatten -> ReLU -> 2d pooling -> dropout
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return torch.sigmoid(self.fc2(x)) # necessary to output probability between 0 to 1

#### CNN (separate HSI/MSI models): helper functions
- cnn_train: Adam optimizer, binary cross entropy loss metric measurement
- cnn_evaluate

In [ ]:
def cnn_train(model, train_loader, lr, epochs=3):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss() # binary cross entropy loss metric measurement
    history = {'loss': [], 'accuracy': []} 
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, batch_labels in train_loader:
            images, batch_labels = images.to(device), batch_labels.to(device)
            optimizer.zero_grad() # isolate current from prev batches; gradients accumulate by default
            outputs = model(images) # forward pass
            loss = criterion(outputs, batch_labels) # compare predictions to ground truth
            loss.backward() # backward pass
            optimizer.step() # determined by learning rate
            running_loss += loss.item()
            predictions = (outputs > 0.5).float() # creates class division (healthy: 0, tumor: 1)
            correct += (predictions == batch_labels).sum().item()
            total += batch_labels.size(0)    
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = correct / total
        history['loss'].append(epoch_loss)
        history['accuracy'].append(epoch_acc)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")
    return history
    
def cnn_evaluate(model, val_loader):
    model.eval() # no batch normalization and no dropout (training trick)
    correct = 0
    total = 0
    with torch.no_grad(): # speed up GPU, saves memory
        for images, batch_labels in val_loader:
            images, batch_labels = images.to(device), batch_labels.to(device)
            outputs = model(images)
            predictions = (outputs > 0.5).float()
            correct += (predictions == batch_labels).sum().item()
            total += batch_labels.size(0)
    return correct / total

#### CNN (separate HSI/MSI models): train-test split

In [ ]:
# 80/20 train-test split
dev_df, final_test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=seed)

# smaller subset for hyperparameter tuning
tune_df, _ = train_test_split(dev_df, train_size=0.5, stratify=dev_df["label"], random_state=seed)

#### CNN (separate HSI/MSI models): hyperparameter tuning
- variable learning rates and dropouts

In [ ]:
learning_rates = [0.001, 0.002, 0.003]
dropouts = [0.2, 0.3, 0.4]
epochs = 3 # constant across all models for consistency
batch = 16
labels = tune_df["label"].values
indices = np.arange(len(tune_df))
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)

# best hyperparameters tracker
best_score_hsi = 0
best_score_msi = 0
best_params_hsi = None
best_params_msi = None

In [ ]:
results = {(lr, dr): {'hsi': [], 'msi': []} for lr in learning_rates for dr in dropouts} # dictionary to store hyperparameter tuning results

for fold, (train_ind, val_ind) in enumerate(skf.split(indices, labels)): # train/evaluate all combinations on a per-fold basis
    print(f"\n FOLD {fold+1}/{skf.n_splits}")
    
    train_df, val_df = tune_df.iloc[train_ind], tune_df.iloc[val_ind]

    # HSI only PyTorch dataset
    ds_hsi_train = KidneyDataset(train_df["processed_path_hsi"].values, train_df["label"].values)
    ds_hsi_val = KidneyDataset(val_df["processed_path_hsi"].values, val_df["label"].values)
    train_loader_hsi = DataLoader(ds_hsi_train, batch_size=batch, shuffle=True, num_workers=0)
    val_loader_hsi = DataLoader(ds_hsi_val, batch_size=batch, shuffle=False, num_workers=0) 

    # MSI only PyTorch dataset
    ds_msi_train = KidneyDataset(train_df["processed_path_msi"].values, train_df["label"].values)
    ds_msi_val = KidneyDataset(val_df["processed_path_msi"].values, val_df["label"].values)
    train_loader_msi = DataLoader(ds_msi_train, batch_size=batch, shuffle=True, num_workers=0)
    val_loader_msi = DataLoader(ds_msi_val, batch_size=batch, shuffle=False, num_workers=0)

    for lr in learning_rates:
        for dr in dropouts:
            print(f"Testing LR: {lr}, DR: {dr}")
            
            # HSI Model Training
            model_hsi = CNN(dropout_rate=dr).to(device)
            cnn_train(model_hsi, train_loader_hsi, lr, epochs=epochs)
            val_acc_hsi = cnn_evaluate(model_hsi, val_loader_hsi)
            results[(lr, dr)]['hsi'].append(val_acc_hsi)
            
            # MSI Model Training
            model_msi = CNN(dropout_rate=dr).to(device)
            cnn_train(model_msi, train_loader_msi, lr, epochs=epochs)
            val_acc_msi = cnn_evaluate(model_msi, val_loader_msi)
            results[(lr, dr)]['msi'].append(val_acc_msi)

            # memory cleanup
            del model_hsi, model_msi
            gc.collect()

print("\n Hyperparameter Tuning Results")
for (lr, dr), scores in results.items():
    mean_hsi = np.mean(scores['hsi'])
    mean_msi = np.mean(scores['msi'])
    print(f"LR: {lr} | DR: {dr} | Mean HSI: {mean_hsi:.4f} | Mean MSI: {mean_msi:.4f}")
    if mean_hsi > best_score_hsi: # track best parameters
        best_score_hsi = mean_hsi
        best_params_hsi = (lr, dr)
    if mean_msi > best_score_msi:
        best_score_msi = mean_msi
        best_params_msi = (lr, dr)

#### CNN (separate HSI/MSI models): hyperparameter tuning results
- HSI: learning rate 0.001, dropout 0.3
- MSI: learning rate 0.002, dropout 0.3

In [ ]:
# see results of hyperparameter tuning
print(best_score_hsi)
print(best_score_msi)
print(best_params_hsi)
print(best_params_msi)

#### CNN (separate HSI/MSI models): evaluate on final test set
- use optimized hyperparameters after fine tuning

In [ ]:
epochs = 3

lr_hsi, dr_hsi = best_params_hsi 
lr_msi, dr_msi = best_params_msi  

# HSI only
print("\n CNN: HSI only model")
ds_hsi_final_train = KidneyDataset(dev_df["processed_path_hsi"].values, labels=dev_df["label"].values)
ds_hsi_final_test = KidneyDataset(final_test_df["processed_path_hsi"].values, labels=final_test_df["label"].values)
loader_hsi_final_train = DataLoader(ds_hsi_final_train, batch_size=batch, shuffle=True)
loader_hsi_final_test = DataLoader(ds_hsi_final_test, batch_size=batch, shuffle=False)

final_model_hsi = CNN(dropout_rate=dr_hsi).to(device)
history_hsi_dict = cnn_train(final_model_hsi, loader_hsi_final_train, lr_hsi, epochs=epochs)
test_acc_hsi = cnn_evaluate(final_model_hsi, loader_hsi_final_test)
print(f"Final test accuracy (CNN: HSI only model): {test_acc_hsi:.4f}")

# MSI only
print("\n CNN: MSI only model")
ds_msi_final_train = KidneyDataset(dev_df["processed_path_msi"].values, labels=dev_df["label"].values)
ds_msi_final_test = KidneyDataset(final_test_df["processed_path_msi"].values, labels=final_test_df["label"].values)
loader_msi_final_train = DataLoader(ds_msi_final_train, batch_size=batch, shuffle=True)
loader_msi_final_test = DataLoader(ds_msi_final_test, batch_size=batch, shuffle=False)

final_model_msi = CNN(dropout_rate=dr_msi).to(device)
history_msi_dict = cnn_train(final_model_msi, loader_msi_final_train, lr_msi, epochs=epochs)
test_acc_msi = cnn_evaluate(final_model_msi, loader_msi_final_test)
print(f"Final test accuracy (CNN: MSI only model): {test_acc_msi:.4f}")

In [ ]:
# save logs to CSV
pd.DataFrame(history_hsi_dict).to_csv("history_cnn_hsi_only.csv", index=False)
pd.DataFrame(history_msi_dict).to_csv("history_cnn_msi_only.csv", index=False)

# save weights
torch.save(final_model_hsi.state_dict(), "final_cnn_hsi_only.pth")
torch.save(final_model_msi.state_dict(), "final_cnn_msi_only.pth")

#### CNN (separate HSI/MSI models): confusion matrices and classification report

In [ ]:
# CNN (HSI only model) - confusion matrix + classification report
final_model_hsi = CNN(dropout_rate=0.3).to(device)
final_model_hsi.load_state_dict(torch.load("final_cnn_hsi_only.pth", map_location=device)) # loading saved weights
final_model_hsi.eval()
cnn_separate_metrics(final_model_hsi, loader_hsi_final_test, "CNN (HSI only)")

# CNN (MSI only model) - confusion matrix + classification report
final_model_msi = CNN(dropout_rate=0.3).to(device)
final_model_msi.load_state_dict(torch.load("final_cnn_msi_only.pth", map_location=device)) # loading saved weights
final_model_msi.eval()
cnn_separate_metrics(final_model_msi, loader_msi_final_test, "CNN (MSI only)")

### CNN (late fusion)
- separate HSI and MSI branches to yield different flatten vectors
- pool individual decisions to create a "weighted" final decision

In [ ]:
# CNN (late fusion): Y-shaped input 
class LateFusionCNN(nn.Module):
    
    # define all CNN layers
    def __init__(self, dropout_rate=0.3):
        super(LateFusionCNN, self).__init__()
        self.hsi_conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.hsi_bn1 = nn.BatchNorm2d(16)
        self.hsi_conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # "hsi_final_conv" for Grad-CAM
        self.hsi_bn2 = nn.BatchNorm2d(32)
        self.msi_conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.msi_bn1 = nn.BatchNorm2d(16)
        self.msi_conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # "msi_final_conv" for Grad-CAM
        self.msi_bn2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.drop = nn.Dropout(dropout_rate)
        self.fuse_bn = nn.BatchNorm1d(32*32*32*2) # fuse HSI and MSI
        self.fc1 = nn.Linear(32*32*32*2, 32) # concatenate MSI and HSI channels, each 32 channels*32*32 (after 2 pools)
        self.fc2 = nn.Linear(32, 1)

    # data flow from/to defined CNN layers
    def forward(self, x_hsi, x_msi):
        h = self.pool(F.relu(self.hsi_bn1(self.hsi_conv1(x_hsi)))) # HSI layers
        h = self.drop(h)
        h = self.pool(F.relu(self.hsi_bn2(self.hsi_conv2(h))))
        h = torch.flatten(h, 1)
        m = self.pool(F.relu(self.msi_bn1(self.msi_conv1(x_msi)))) # MSI layers (identical to MSI)
        m = self.drop(m)
        m = self.pool(F.relu(self.msi_bn2(self.msi_conv2(m))))
        m = torch.flatten(m, 1)
        combined = torch.cat((h, m), dim=1) # fuse HSI and mSI
        combined = self.fuse_bn(combined) 
        z = F.relu(self.fc1(combined))
        z = self.drop(z)
        return torch.sigmoid(self.fc2(z)) # sigmoid activation necessary for binary classification

#### CNN (late fusion): helper functions

In [ ]:
# CNN (late fusion) model training and metric tracking
def fusion_train(model, loader, lr, epochs=3):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss() # same optimizer & criterion as CNN (HSI/MSI only models) for consistency
    history = {'loss': [], 'accuracy': []} 
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0       
        for img_hsi, img_msi, batch_labels in loader:
            img_hsi, img_msi, batch_labels = img_hsi.to(device), img_msi.to(device), batch_labels.to(device)    
            optimizer.zero_grad()
            outputs = model(img_hsi, img_msi) # dual HSI and MSI input
            loss = criterion(outputs, batch_labels)
            loss.backward() # backpropagation for weight updates
            optimizer.step() # stepping down the gradient
            running_loss += loss.item()
            predictions = (outputs > 0.5).float()
            correct += (predictions == batch_labels).sum().item()
            total += batch_labels.size(0)
        epoch_loss = running_loss / len(loader)
        epoch_acc = correct / total
        history['loss'].append(epoch_loss)
        history['accuracy'].append(epoch_acc)    
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")      
    return history

# CNN (late fusion) model evaluation
def fusion_evaluate(model, loader):
    model.eval() 
    correct = 0
    total = 0
    with torch.no_grad(): # speeds up GPU
        for img_hsi, img_msi, batch_labels in loader:
            img_hsi, img_msi, batch_labels = img_hsi.to(device), img_msi.to(device), batch_labels.to(device)
            outputs = model(img_hsi, img_msi) # dual HSI and MSI input
            predictions = (outputs > 0.5).float()
            correct += (predictions == batch_labels).sum().item()
            total += batch_labels.size(0)
    return correct / total

#### CNN (late fusion): hyperparameter tuning
- informed hyperparameter selection

In [ ]:
hsi_lr, hsi_dr = best_params_hsi
msi_lr, msi_dr = best_params_msi

# smaller focused list for hyperparameter tuning (compared to the list for CNN HSI-only and MSI-only hyperparameter optimization)
fusion_lrs = list(set([hsi_lr, msi_lr]))
fusion_drs = list(set([hsi_dr, msi_dr]))

best_score_fusion = 0
best_params_fusion = None

In [ ]:
epochs = 3 # for consistency
results_fusion = {(lr, dr): {'fusion': []} for lr in fusion_lrs for dr in fusion_drs} # dictionary to store results

for lr in fusion_lrs:
    for dr in fusion_drs:
        print(f"\n CNN (late fusion) | LR: {lr} | DR: {dr}")
        for fold, (train_ind, val_ind) in enumerate(skf.split(indices, tune_df["label"].values)):
            print(f" Fold {fold+1}\n")
            train_df = tune_df.iloc[train_ind]
            val_df = tune_df.iloc[val_ind]

            # PyTorch dual dataset
            ds_train = DualKidneyDataset(train_df["processed_path_hsi"].values, train_df["processed_path_msi"].values, train_df["label"].values)
            ds_val = DualKidneyDataset(val_df["processed_path_hsi"].values, val_df["processed_path_msi"].values, val_df["label"].values)
            train_loader = DataLoader(ds_train, batch_size=batch, shuffle=True)
            val_loader = DataLoader(ds_val, batch_size=batch, shuffle=False)

            # create, train, evaluate late fusion CNN model
            model_fusion = LateFusionCNN(dropout_rate=dr).to(device)
            history = fusion_train(model_fusion, train_loader, lr, epochs=epochs)
            val_acc = fusion_evaluate(model_fusion, val_loader)
            results_fusion[(lr, dr)]['fusion'].append(val_acc)
            print(f"Accuracy: {val_acc:.4f}")
            
            # cleanup for faster run
            del model_fusion
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        mean_fusion = np.mean(results_fusion[(lr, dr)]['fusion'])
        std_fusion = np.std(results_fusion[(lr, dr)]['fusion'])
        print(f"Hyperparamater tuning results: LR: {lr}, DR: {dr}, Mean Acc: {mean_fusion:.4f} (+/- {std_fusion:.4f})")
        
        if mean_fusion > best_score_fusion: # update to best parameters as needed
            best_score_fusion = mean_fusion
            best_params_fusion = (lr, dr)

#### CNN (late fusion): hyperparameter tuning results
- see results of hyperparameter tuning

In [ ]:
print(best_score_fusion)
print(best_params_fusion)

#### CNN (late fusion): evaluate on final test set

In [ ]:
lr_fusion, dr_fusion = best_params_fusion 
ds_fusion_final_train = DualKidneyDataset(dev_df["processed_path_hsi"].values, dev_df["processed_path_msi"].values, dev_df["label"].values)
ds_fusion_final_test = DualKidneyDataset(final_test_df["processed_path_hsi"].values, final_test_df["processed_path_msi"].values, 
                                         final_test_df["label"].values)

loader_fusion_final_train = DataLoader(ds_fusion_final_train, batch_size=batch, shuffle=True) # full 80% of dataset
loader_fusion_final_test = DataLoader(ds_fusion_final_test, batch_size=batch, shuffle=False) # held out test set (20%)

# create, train, evaluate CNN (late fusion) model
final_model_fusion = LateFusionCNN(dropout_rate=dr_fusion).to(device)
history_fusion_dict = fusion_train(final_model_fusion, loader_fusion_final_train, lr_fusion, epochs=epochs)
test_acc_fusion = fusion_evaluate(final_model_fusion, loader_fusion_final_test)

print(f"CNN (late fusion) Test Accuracy: {test_acc_fusion:.4f}")

In [ ]:
# save logs to CSV
pd.DataFrame(history_fusion_dict).to_csv("history_cnn_late_fusion.csv", index=False)

# save weights
torch.save(final_model_fusion.state_dict(), "final_cnn_late_fusion.pth")

#### CNN (late fusion): classification report and confusion matrices

In [ ]:
# CNN (late fusion) -- for visualizing confusion matrices and classification report
class KidneyFusionDataset(torch.utils.data.Dataset):
    def __init__(self, hsi_paths, msi_paths, labels, transform=None):
        self.hsi_paths = hsi_paths
        self.msi_paths = msi_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_hsi = cv2.cvtColor(cv2.imread(self.hsi_paths[idx]), cv2.COLOR_BGR2RGB) 
        img_msi = cv2.cvtColor(cv2.imread(self.msi_paths[idx]), cv2.COLOR_BGR2RGB)
        img_hsi = cv2.resize(img_hsi, (img_size[0], img_size[1])) / 255.0 # normalize
        img_msi = cv2.resize(img_msi, (img_size[0], img_size[1])) / 255.0
        img_hsi = torch.from_numpy(img_hsi).permute(2,0,1).float() # (channels, height, width)
        img_msi = torch.from_numpy(img_msi).permute(2,0,1).float() 
        label = torch.tensor(self.labels[idx], dtype=torch.float32).reshape(1)
        return img_hsi, img_msi, label

In [ ]:
ds_fusion_final_test = KidneyFusionDataset(hsi_paths=final_test_df["processed_path_hsi"].values,
                                           msi_paths=final_test_df["processed_path_msi"].values,
                                           labels=final_test_df["label"].values)

loader_fusion_final_test = torch.utils.data.DataLoader(ds_fusion_final_test, batch_size=batch, shuffle=False)

In [ ]:
final_model_fusion = LateFusionCNN(dropout_rate=0.3).to(device)
final_model_fusion.load_state_dict(torch.load("final_cnn_late_fusion.pth", map_location=device))
final_model_fusion.eval()

cnn_fusion_metrics(final_model_fusion, loader_fusion_final_test, "CNN (late fusion)")

### MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size=(img_size[0], img_size[1]), channels=3, dropout_rate=0.3):
        super(MLP, self).__init__()
        self.input_dim = input_size[0] * input_size[1] * channels

        # define all MLP layers
        self.main = nn.Sequential(nn.Flatten(), # (channels, height, width) flattened into 1D tensor
                                  nn.Linear(self.input_dim, 256), # each pixel connected to 256 neurons
                                  nn.ReLU(),
                                  nn.Dropout(dropout_rate), # to prevent overfitting       
                                  nn.Linear(256, 64),
                                  nn.ReLU(),
                                  nn.Linear(64, 1),
                                  nn.Sigmoid()) # for binary classification (number between 0 and 1)
    
    def forward(self, x):
        if isinstance(x, dict): x = x['input_hsi'] # safety mechanism to extract appropriate tensor
        return self.main(x)

In [ ]:
mlp_lrs = [0.001, 0.002, 0.003]
mlp_drs = [0.2, 0.3, 0.4]
epochs = 3 
batch = 16

best_score_hsi = 0
best_params_hsi = None
best_score_msi = 0
best_params_msi = None

results_mlp = {(lr, dr): {'hsi': [], 'msi': []} for lr in mlp_lrs for dr in mlp_drs}

for fold, (train_ind, val_ind) in enumerate(skf.split(indices, labels)):
    print(f"\nFOLD {fold+1}/{skf.n_splits}")
    
    train_df, val_df = tune_df.iloc[train_ind], tune_df.iloc[val_ind]

    # HSI only
    ds_hsi_train = KidneyDataset(train_df["processed_path_hsi"].values, train_df["label"].values)
    ds_hsi_val = KidneyDataset(val_df["processed_path_hsi"].values, val_df["label"].values)
    loader_hsi_train = DataLoader(ds_hsi_train, batch_size=batch, shuffle=True)
    loader_hsi_val = DataLoader(ds_hsi_val, batch_size=batch, shuffle=False)

    # MSI only
    ds_msi_train = KidneyDataset(train_df["processed_path_msi"].values, train_df["label"].values)
    ds_msi_val = KidneyDataset(val_df["processed_path_msi"].values, val_df["label"].values)
    loader_msi_train = DataLoader(ds_msi_train, batch_size=batch, shuffle=True)
    loader_msi_val = DataLoader(ds_msi_val, batch_size=batch, shuffle=False)
    
    for lr in mlp_lrs:
        for dr in mlp_drs:
            print(f"LR: {lr}, DR: {dr}")
            mlp_hsi_model = MLP(dropout_rate=dr).to(device)
            cnn_train(mlp_hsi_model, loader_hsi_train, lr, epochs=epochs)
            acc_hsi = cnn_evaluate(mlp_hsi_model, loader_hsi_val)
            results_mlp[(lr, dr)]['hsi'].append(acc_hsi)
            mlp_msi_model = MLP(dropout_rate=dr).to(device)
            cnn_train(mlp_msi_model, loader_msi_train, lr, epochs=epochs)
            acc_msi = cnn_evaluate(mlp_msi_model, loader_msi_val)
            results_mlp[(lr, dr)]['msi'].append(acc_msi)
            
            # cleanup for faster run
            del mlp_hsi_model, mlp_msi_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

print("\n MLP hyperparameter tuning results (summary)")
for (lr, dr), scores in results_mlp.items():
    mean_hsi = np.mean(scores['hsi'])
    mean_msi = np.mean(scores['msi'])
    print(f"LR: {lr} | DR: {dr} | HSI Acc: {mean_hsi:.4f} | MSI Acc: {mean_msi:.4f}")
    
    if mean_hsi > best_score_hsi:
        best_score_hsi = mean_hsi
        best_params_hsi = (lr, dr)
    if mean_msi > best_score_msi:
        best_score_msi = mean_msi
        best_params_msi = (lr, dr)

print(f"\nMLP HSI parameters: {best_params_hsi} (Acc: {best_score_hsi:.4f})")
print(f"\nMLP MSI parameters: {best_params_msi} (Acc: {best_score_msi:.4f})")

In [ ]:
lr_mlp_hsi, dr_mlp_hsi = best_params_hsi
lr_mlp_msi, dr_mlp_msi = best_params_msi

# initialize MLP HSI only model
ds_mlp_hsi_final_train = KidneyDataset(dev_df["processed_path_hsi"].values, dev_df["label"].values)
ds_mlp_hsi_final_test = KidneyDataset(final_test_df["processed_path_hsi"].values, final_test_df["label"].values)
loader_mlp_hsi_final_train = DataLoader(ds_mlp_hsi_final_train, batch_size=batch, shuffle=True)
loader_mlp_hsi_final_test = DataLoader(ds_mlp_hsi_final_test, batch_size=batch, shuffle=False)

# train and test MLP HSI only model
final_model_mlp_hsi = MLP(dropout_rate=dr_mlp_hsi).to(device)
cnn_train(final_model_mlp_hsi, loader_mlp_hsi_final_train, lr_mlp_hsi, epochs=epochs)
test_acc_mlp_hsi = cnn_evaluate(final_model_mlp_hsi, loader_mlp_hsi_final_test)
print(f"Test accuracy (MLP HSI only): {test_acc_mlp_hsi:.4f}")


# initialize MLP MSI only model
ds_mlp_msi_final_train = KidneyDataset(dev_df["processed_path_msi"].values, dev_df["label"].values)
ds_mlp_msi_final_test = KidneyDataset(final_test_df["processed_path_msi"].values, final_test_df["label"].values)
loader_mlp_msi_final_train = DataLoader(ds_mlp_msi_final_train, batch_size=batch, shuffle=True)
loader_mlp_msi_final_test = DataLoader(ds_mlp_msi_final_test, batch_size=batch, shuffle=False)

# train and test MLP MSI only model
final_model_mlp_msi = MLP(dropout_rate=dr_mlp_msi).to(device)
cnn_train(final_model_mlp_msi, loader_mlp_msi_final_train, lr_mlp_msi, epochs=epochs)
test_acc_mlp_msi = cnn_evaluate(final_model_mlp_msi, loader_mlp_msi_final_test)
print(f"Test accuracy (MLP MSI only): {test_acc_mlp_msi:.4f}")

torch.save(final_model_mlp_hsi.state_dict(), "final_mlp_hsi_model.pth") # save weights
torch.save(final_model_mlp_msi.state_dict(), "final_mlp_msi_model.pth")

In [ ]:
cnn_separate_metrics(final_model_mlp_hsi, loader_mlp_hsi_final_test, "MLP (HSI only)") # confusion matrix + classification report
cnn_separate_metrics(final_model_mlp_msi, loader_mlp_msi_final_test, "MLP (MSI only)")

#### MLP (late fusion)

In [ ]:
class LateFusionMLP(nn.Module):
    
    def __init__(self, hsi_dim, msi_dim, dropout_rate=0.3):
        super(LateFusionMLP, self).__init__()
        self.hsi_branch = nn.Sequential(nn.Flatten(), # define all MLP layers (for HSI only model)
                                        nn.Linear(hsi_dim, 256),
                                        nn.ReLU(), # ReLU activation introduces nonlinearity
                                        nn.Dropout(dropout_rate),
                                        nn.Linear(256, 64),
                                        nn.ReLU())
        self.msi_branch = nn.Sequential(nn.Flatten(), # define all MLP layers (for MSI only model)
                                        nn.Linear(msi_dim, 256),
                                        nn.ReLU(),
                                        nn.Dropout(dropout_rate),
                                        nn.Linear(256, 64),
                                        nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(128, 32), # fuse HSI and MSI models at the end
                                        nn.ReLU(),
                                        nn.Linear(32, 1),
                                        nn.Sigmoid())
        
    def forward(self, x_hsi, x_msi):
        feat_hsi = self.hsi_branch(x_hsi)
        feat_msi = self.msi_branch(x_msi)
        combined = torch.cat((feat_hsi, feat_msi), dim=1) # concatenate HSI & MSI features along feature axis
        return self.classifier(combined)

In [ ]:
fusion_lrs = list(set([best_params_hsi[0], best_params_msi[0]])) 
fusion_drs = list(set([best_params_hsi[1], best_params_msi[1]]))

best_score_mlp_fusion = 0
best_params_mlp_fusion = None
results_mlp_fusion = {(lr, dr): [] for lr in fusion_lrs for dr in fusion_drs}

hsi_input_dim = img_size[0] * img_size[1] * 3 # 3 channels in RGB images
msi_input_dim = img_size[0] * img_size[1] * 3

for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
    print(f"\n FOLD {fold+1} ---")
    
    train_df, val_df = tune_df.iloc[train_idx], tune_df.iloc[val_idx]

    ds_fusion_train = DualKidneyDataset(train_df["processed_path_hsi"].values, train_df["processed_path_msi"].values, train_df["label"].values)
    ds_fusion_val = DualKidneyDataset(val_df["processed_path_hsi"].values, val_df["processed_path_msi"].values, val_df["label"].values)
    loader_train = DataLoader(ds_fusion_train, batch_size=batch, shuffle=True)
    loader_val = DataLoader(ds_fusion_val, batch_size=batch, shuffle=False)

    for lr in fusion_lrs:
        for dr in fusion_drs:
            print(f"LR: {lr}, DR: {dr}")
            
            model_fusion = LateFusionMLP(hsi_input_dim, msi_input_dim, dropout_rate=dr).to(device)
            fusion_train(model_fusion, loader_train, lr, epochs=epochs)
            acc = fusion_evaluate(model_fusion, loader_val)
            results_mlp_fusion[(lr, dr)].append(acc)

            # cleanup for faster run
            del model_fusion
            if torch.cuda.is_available(): 
                torch.cuda.empty_cache()
            gc.collect()

# collect and store hyperparameter optimization results
for (lr, dr), scores in results_mlp_fusion.items():
    mean_acc = np.mean(scores)
    if mean_acc > best_score_mlp_fusion:
        best_score_mlp_fusion = mean_acc
        best_params_mlp_fusion = (lr, dr)

print(f"\nMLP (late fusion) hyperparameter tuning results: LR={best_params_mlp_fusion[0]}, DR={best_params_mlp_fusion[1]} (accuracy: {best_score_mlp_fusion:.4f})")

In [ ]:
lr_mlp_f, dr_mlp_f = best_params_mlp_fusion

ds_mlp_fusion_final_train = DualKidneyDataset(dev_df["processed_path_hsi"].values, dev_df["processed_path_msi"].values, dev_df["label"].values)
ds_mlp_fusion_final_test = DualKidneyDataset(final_test_df["processed_path_hsi"].values, final_test_df["processed_path_msi"].values, 
                                             final_test_df["label"].values)
loader_mlp_fusion_final_train = DataLoader(ds_mlp_fusion_final_train, batch_size=batch, shuffle=True)
loader_mlp_fusion_final_test = DataLoader(ds_mlp_fusion_final_test, batch_size=batch, shuffle=False)

hsi_in = img_size[0] * img_size[1] * 3
msi_in = img_size[0] * img_size[1] * 3

final_mlp_fusion_model = LateFusionMLP(hsi_in, msi_in, dropout_rate=dr_mlp_f).to(device)
fusion_train(final_mlp_fusion_model, loader_mlp_fusion_final_train, lr_mlp_f, epochs=epochs) # training model
torch.save(final_mlp_fusion_model.state_dict(), "final_mlp_fusion_model.pth") # save weights
final_test_acc_fusion = fusion_evaluate(final_mlp_fusion_model, loader_mlp_fusion_final_test) # evaluate model on held out test set
print(f"\nTest accuracy (MLP late fusion): {final_test_acc_fusion:.4f}")

cnn_fusion_metrics(final_mlp_fusion_model, loader_mlp_fusion_final_test, "Final Late Fusion MLP") # confusion matrix + classification report

### SVM
- HSI only and MSI only models: 14 features per image (10 LBP, 4 GLCM)
- early fusion: concatenate 14 HSI only features and 14 MSI only features for a total of 28 features
- stark contrast in number of features compared to CNN
- comparing SVM and CNN frames the question: can simple features (what humans see) detect kidney tumors as well as deep learning (complex)?

#### SVM: helper functions
- extract_single_texture: extract LBP and GLCM features from one image
- get_fused_svm_data: extract features for all images in given data

In [ ]:
# extract 14 features per image
def extract_single_texture(path, img_size=(img_size[0], img_size[1])):
    import cv2 # imports included inside function for parallel workers (faster)
    import numpy as np
    from skimage.feature import local_binary_pattern
    from skimage.color import rgb2gray

    # some tricky imports; dependent on imported version
    try:
        from skimage.feature import graycomatrix, graycoprops
    except ImportError: 
        from skimage.feature import greycomatrix as graycomatrix
        from skimage.feature import greycoprops as graycoprops

    image = cv2.imread(path)
    if image is None: # safety net
        return np.zeros(14)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, img_size)
    gray = (rgb2gray(image)*255).astype(np.uint8) # convert to greyscale for GLCM
    
    # LBP (local binary patterns): see/quantify texture (locally)
    lbp = local_binary_pattern(gray, 8, 1, method='uniform') 
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    
    # Haralick, or GLCM (grey level co-occurence matrix): see/compare pixel pairs (globally)
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True) # 256 levels for 8-bit image
    glcm_feats = np.array([ graycoprops(glcm, 'contrast')[0, 0], # four features of interest
                            graycoprops(glcm, 'dissimilarity')[0, 0],
                            graycoprops(glcm, 'homogeneity')[0, 0],
                            graycoprops(glcm, 'energy')[0, 0] ])
    return np.concatenate([lbp_hist, glcm_feats]) # 14 features total; 10 from LBP, 4 from Haralick/GLCM

# extract features across dataset
def get_fused_svm_data(df):
    # Early Fusion: Concatenating HSI texture and MSI texture vectors
    hsi_feats = Parallel(n_jobs=-1)(delayed(extract_single_texture)(p) for p in df["processed_path_hsi"].values) # n_jobs = -1: can run in parallel
    msi_feats = Parallel(n_jobs=-1)(delayed(extract_single_texture)(p) for p in df["processed_path_msi"].values)
    return np.hstack([np.array(hsi_feats), np.array(msi_feats)])

#### SVM: hyperparameter tuning

In [ ]:
X_all = get_fused_svm_data(tune_df) # early fusion SVM
y_tune = tune_df["label"].values
datasets = { "SVM (HSI only)": X_all[:, :14], # first 14 features are from the HSI image
             "SVM (MSI only)": X_all[:, 14:], # last 14 features are from the corresponding MSI image
             "SVM (early fusion)": X_all }
param_grid = [ {'kernel': ['linear'], 'C': [0.001, 0.01, 0.1]},
               {'kernel': ['rbf'], 'C': [0.001, 0.01, 0.1], 'gamma': [0.01, 0.1, 'scale']},
               {'kernel': ['poly'], 'C': [0.001, 0.01, 0.1], 'degree': [2, 3], 'gamma': ['scale']} ]

all_combos = []
for entry in param_grid:
    all_combos.extend(list(ParameterGrid(entry)))

best_results = {} # to store the results of hyperparameter tuning
for name, X_tune in datasets.items():
    print(f"\n Hyperparameter tuning: {name}")
    best_score_svm = 0
    best_params_svm = None
    combo_scores = {i: [] for i in range(len(all_combos))}

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_tune, y_tune)):
        print(f"\n{name}: Fold {fold + 1}/3")
        X_tr, X_va = X_tune[train_idx], X_tune[val_idx]
        y_tr, y_va = y_tune[train_idx], y_tune[val_idx]
        sc = StandardScaler() # zero mean and unit variance to place LBP and GLCM on the same scale
        X_tr_scaled = sc.fit_transform(X_tr)
        X_va_scaled = sc.transform(X_va)
        
        for i, params in enumerate(tqdm(all_combos, desc=f"Fold {fold+1}")):
            clf = SVC(**params, random_state=seed)
            clf.fit(X_tr_scaled, y_tr)
            preds = clf.predict(X_va_scaled)
            acc = accuracy_score(y_va, preds)
            combo_scores[i].append(acc)
            
    for i, params in enumerate(all_combos): # save best hyperparameters
        mean_acc = np.mean(combo_scores[i])
        if mean_acc > best_score_svm:
            best_score_svm = mean_acc
            best_params_svm = params
            
    best_results[name] = {"params": best_params_svm, "score": best_score_svm}
    print(f"Hyperparameter Tuning Results ({name}): {best_params_svm} | Acc: {best_score_svm:.4f}")

print("List of hyperparameter tuning results") # summary of optimized hyperparameters for all SVM models
for name, res in best_results.items():
    print(f"{name:<15}: {res['score']:.4f} from {res['params']}")

#### SVM: evaluate on final held out test set

In [ ]:
# evaluating on final held out test set

X_dev_all = get_fused_svm_data(dev_df)
X_test_all = get_fused_svm_data(final_test_df)
y_train_final = dev_df["label"].values
y_test_final = final_test_df["label"].values

modality_slices = {"SVM (HSI only)": (0, 14), # first 14 features
                   "SVM (MSI only)": (14, 28), # last 14 features
                   "SVM (early fusion)": (0, 28)} # concatenate HSI and MSI features
final_reports = {}

for name, (start, end) in modality_slices.items():
    X_tr = X_dev_all[:, start:end] # train
    X_te = X_test_all[:, start:end] # test
    best_p = best_results[name]["params"] # optimized hyperparameters
    final_sc = StandardScaler()
    X_tr_scaled = final_sc.fit_transform(X_tr)
    X_te_scaled = final_sc.transform(X_te)

    # train and evaluate (predict) on held out test set
    model = SVC(**best_p, random_state=seed)
    model.fit(X_tr_scaled, y_train_final)
    test_preds = model.predict(X_te_scaled)
    acc = accuracy_score(y_test_final, test_preds)
    final_reports[name] = acc
    
    print(f"\n{name} (Parameters: {best_p})")
    print(classification_report(y_test_final, test_preds, digits=6))

for name, acc in final_reports.items():
    print(f"{name:<15}: {acc:.6f}")

In [ ]:
# save model and scaler
tuning_df = pd.DataFrame(best_results).T 
tuning_df.to_csv("svm_tuning_results_comparison.csv", index=True)

for name in ["SVM (HSI only)", "SVM (MSI only)", "SVM (early fusion)"]:
    file_name = name.lower().replace("-", "_").replace(" ", "_")
    if name == "SVM (HSI only)":
        X_tr = X_dev_all[:, :14]
    elif name == "SVM (MSI only)":
        X_tr = X_dev_all[:, 14:]
    else:
        X_tr = X_dev_all
    best_p = best_results[name]["params"]
    sc = StandardScaler()
    X_scaled = sc.fit_transform(X_tr)
    
    model = SVC(**best_p, random_state=seed)
    model.fit(X_scaled, y_train_final)
    joblib.dump(model, f"final_svm_{file_name}.joblib")
    joblib.dump(sc, f"scaler_svm_{file_name}.pkl")

### Heat Maps

In [ ]:
final_model_hsi = CNN()
final_model_hsi.to(device)

final_model_msi = CNN()
final_model_msi.to(device)

final_model_fusion = LateFusionCNN()
final_model_fusion.to(device)

cnn_hsi_checkpoint = torch.load('final_cnn_hsi_only.pth', map_location=device)
final_model_hsi.load_state_dict(cnn_hsi_checkpoint)

cnn_msi_checkpoint = torch.load('final_cnn_msi_only.pth', map_location=device)
final_model_msi.load_state_dict(cnn_msi_checkpoint)

cnn_fusion_checkpoint = torch.load('final_cnn_late_fusion.pth', map_location=device)
final_model_fusion.load_state_dict(cnn_fusion_checkpoint)

final_model_hsi.eval()
final_model_msi.eval()
final_model_fusion.eval()

In [ ]:
def cnn_gradcam(model, input_tensor, original_image, target_layer): # converts original image to grayscale for better visualization
    target_layers = [target_layer]
    cam = GradCAM(model=model, target_layers=target_layers)
    targets = [ClassifierOutputTarget(0)] 
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets) # generate mask
    gray_img = np.dot(original_image[...,:3], [0.2989, 0.5870, 0.1140]) # convert to grayscale
    # values from https://www.dynamsoft.com/blog/insights/image-processing/image-processing-101-color-space-conversion/ 
    gray_img[gray_img < 0.1] = 0.0 # minimize background noise for better viewability
    gray_img_3ch = np.stack([gray_img] * 3, axis=-1) # stack grayscale onto 3 channels
    plot = show_cam_on_image(gray_img_3ch, grayscale_cam[0, :], use_rgb=True) # overlay heatmap on original (now grayscaled) image
    return plot

In [ ]:
class FusionWrapper(torch.nn.Module):
    def __init__(self, model, msi_tensor):
        super().__init__()
        self.model = model
        self.msi_tensor = msi_tensor

    def forward(self, x_hsi):
        return self.model(x_hsi, self.msi_tensor) # cleverly feeds MSI as well to effectively appear as a single input model

def fusion_gradcam(model, hsi_tensor, msi_tensor, original_image, target_layer):
    wrapped_model = FusionWrapper(model, msi_tensor) # transform into single input model
    wrapped_model.eval() 
    target_layers = [target_layer]
    cam = GradCAM(model=wrapped_model, target_layers=target_layers)
    targets = [ClassifierOutputTarget(0)]
    grayscale_cam = cam(input_tensor=hsi_tensor, targets=targets) # generate mask
    gray_img = np.dot(original_image[...,:3], [0.2989, 0.5870, 0.1140]) # convert to grayscale
    # values from https://www.dynamsoft.com/blog/insights/image-processing/image-processing-101-color-space-conversion/ 
    gray_img[gray_img < 0.1] = 0.0 # minimize background noise for better viewability
    gray_img_3ch = np.stack([gray_img] * 3, axis=-1) # stack grayscale onto 3 channels
    plot = show_cam_on_image(gray_img_3ch, grayscale_cam[0, :], use_rgb=True) # overlay heatmap on original (now grayscaled) image
    return plot

In [ ]:
final_model_hsi = CNN()
final_model_hsi.to(device)

final_model_msi = CNN()
final_model_msi.to(device)

final_model_fusion = LateFusionCNN()
final_model_fusion.to(device)

cnn_hsi_checkpoint = torch.load('final_cnn_hsi_only.pth', map_location=device)
final_model_hsi.load_state_dict(cnn_hsi_checkpoint)

cnn_msi_checkpoint = torch.load('final_cnn_msi_only.pth', map_location=device)
final_model_msi.load_state_dict(cnn_msi_checkpoint)

cnn_fusion_checkpoint = torch.load('final_cnn_late_fusion.pth', map_location=device)
final_model_fusion.load_state_dict(cnn_fusion_checkpoint)

final_model_hsi.eval()
final_model_msi.eval()
final_model_fusion.eval()

final_svm_hsi_only = joblib.load("final_svm_svm_(hsi_only).joblib") # load SVM models
final_svm_msi_only = joblib.load("final_svm_svm_(msi_only).joblib")
final_svm_early_fusion = joblib.load("final_svm_svm_(early_fusion).joblib")

scaler_svm_hsi_only = joblib.load("scaler_svm_svm_(hsi_only).pkl") # load SVM scalers
scaler_svm_msi_only = joblib.load("scaler_svm_svm_(msi_only).pkl")
scaler_svm_early_fusion = joblib.load("scaler_svm_svm_(early_fusion).pkl")

In [ ]:
# recreate predictions from saved SVM models for convenience and efficiency
X_test_fused = get_fused_svm_data(final_test_df) 
y_test = final_test_df['label'].values
X_test_scaled = scaler_svm_early_fusion.transform(X_test_fused)
test_preds = final_svm_early_fusion.predict(X_test_scaled)

In [ ]:
# SVM saliency map (takes in HSI only, MSI only, and early fusion)
def generate_svm_saliency_unified(hsi_path, msi_path, model, scaler, name, window_size=32, stride=8):
    img_h = cv2.cvtColor(cv2.imread(hsi_path), cv2.COLOR_BGR2RGB) # load images
    img_m = cv2.cvtColor(cv2.imread(msi_path), cv2.COLOR_BGR2RGB)
    h, w, _ = img_h.shape
    gray_h = cv2.cvtColor(img_h, cv2.COLOR_RGB2GRAY)
    gray_m = cv2.cvtColor(img_m, cv2.COLOR_RGB2GRAY)
    mask_h = (gray_h > 10)
    mask_m = (gray_m > 10)
    master_mask = np.logical_or(mask_h, mask_m).astype(np.float32) # manually combine HSI & MSI masks for viewability
    heatmap = np.zeros((h, w)) # store probabilities 
    counts = np.zeros((h, w)) # store number of times a window includes a particular pixel
    if "hsi only" in name.lower():
        bg_gray = (gray_h.astype(np.float32) / 255.0) # HSI base for HSI only images
    else:
        bg_gray = (gray_m.astype(np.float32) / 255.0) # MSI base for MSI only images
    bg_img = np.stack([bg_gray] * 3, axis=-1)
    for y in range(0, h - window_size + 1, stride): # sliding window
        for x in range(0, w - window_size + 1, stride):
            patch_h = img_h[y:y+window_size, x:x+window_size]
            patch_m = img_m[y:y+window_size, x:x+window_size]
            if "early fusion" in name.lower(): # get HSI and MSI features
                cv2.imwrite("temp_h.png", cv2.cvtColor(patch_h, cv2.COLOR_RGB2BGR))
                cv2.imwrite("temp_m.png", cv2.cvtColor(patch_m, cv2.COLOR_RGB2BGR))
                feats_h = extract_single_texture("temp_h.png", img_size=(window_size, window_size))
                feats_m = extract_single_texture("temp_m.png", img_size=(window_size, window_size))
                final_feats = np.hstack([feats_h, feats_m]) # stack HSI and MSI features together
            elif "hsi only" in name.lower(): # HSI only
                cv2.imwrite("temp_h.png", cv2.cvtColor(patch_h, cv2.COLOR_RGB2BGR))
                final_feats = extract_single_texture("temp_h.png", img_size=(window_size, window_size))
            else: # MSI only
                cv2.imwrite("temp_m.png", cv2.cvtColor(patch_m, cv2.COLOR_RGB2BGR))
                final_feats = extract_single_texture("temp_m.png", img_size=(window_size, window_size))
            feats_scaled = scaler.transform(final_feats.reshape(1, -1)) # get hyperplane distance (as measure of probability)
            score = model.decision_function(feats_scaled)[0] # raw confidence score
            heatmap[y:y+window_size, x:x+window_size] += score
            counts[y:y+window_size, x:x+window_size] += 1
    heatmap = (heatmap / (counts + 1e-8)) * master_mask # mean
    heatmap = gaussian_filter(heatmap, sigma=2) * master_mask # gaussian filter for visual smoothening
    if np.max(heatmap) > 0: # clipping and normalization
        h_min = np.min(heatmap[master_mask > 0])
        h_max = np.max(heatmap[master_mask > 0])
        heatmap = (heatmap - h_min) / (h_max - h_min + 1e-8)
        heatmap = np.clip(heatmap, 0, 1) * master_mask
    return heatmap, bg_img, master_mask

In [ ]:
test_transform = transforms.Compose([transforms.ToPILImage(), transforms.Resize((img_size[0], img_size[1])), transforms.ToTensor()])

In [ ]:
# reconstruct database from saved weights and files
def build_diagnostic_db(df):
    db = df.copy()
    final_model_hsi.eval()
    final_model_msi.eval()
    final_model_fusion.eval()
    final_model_mlp_hsi.eval()
    final_model_mlp_msi.eval()
    final_mlp_fusion_model.eval()
    c_hsi, c_msi, c_fus = [], [], [] # CNN 
    m_hsi, m_msi, m_fus = [], [], [] # MLP 
    
    with torch.no_grad():
        for i in tqdm(range(len(df)), desc="Neural Nets"):
            h_rgb = cv2.cvtColor(cv2.imread(df.iloc[i]['processed_path_hsi']), cv2.COLOR_BGR2RGB) # load images
            m_rgb = cv2.cvtColor(cv2.imread(df.iloc[i]['processed_path_msi']), cv2.COLOR_BGR2RGB)
            t_h = test_transform(h_rgb).unsqueeze(0).to(device)
            t_m = test_transform(m_rgb).unsqueeze(0).to(device)
            c_hsi.append((torch.sigmoid(final_model_hsi(t_h)) > 0.5).int().item()) # CNN
            c_msi.append((torch.sigmoid(final_model_msi(t_m)) > 0.5).int().item())
            c_fus.append((torch.sigmoid(final_model_fusion(t_h, t_m)) > 0.5).int().item())
            m_hsi.append((torch.sigmoid(final_model_mlp_hsi(t_h)) > 0.5).int().item()) # MLP
            m_msi.append((torch.sigmoid(final_model_mlp_msi(t_m)) > 0.5).int().item())
            m_fus.append((torch.sigmoid(final_mlp_fusion_model(t_h, t_m)) > 0.5).int().item())
    db['cnn_hsi'], db['cnn_msi'], db['cnn_fusion'] = c_hsi, c_msi, c_fus
    db['mlp_hsi'], db['mlp_msi'], db['mlp_fusion'] = m_hsi, m_msi, m_fus

    X_fused = get_fused_svm_data(df) # SVM
    num_hsi = scaler_svm_hsi_only.n_features_in_ # splice fusion features into HSI (first half) and MSI (second half)
    X_hsi = X_fused[:, :num_hsi]
    X_msi = X_fused[:, num_hsi:]
    db['svm_hsi'] = final_svm_hsi_only.predict(scaler_svm_hsi_only.transform(X_hsi))
    db['svm_msi'] = final_svm_msi_only.predict(scaler_svm_msi_only.transform(X_msi))
    db['svm_fusion'] = final_svm_early_fusion.predict(scaler_svm_early_fusion.transform(X_fused))
    
    return db

In [ ]:
# MLP saliency map (takes in HSI only, MSI only, and late fusion)
def generate_mlp_saliency_unified(hsi_path, msi_path, model, mode, window_size=32, stride=8):
    model.eval()
    img_h = cv2.cvtColor(cv2.imread(hsi_path), cv2.COLOR_BGR2RGB) # load images
    img_m = cv2.cvtColor(cv2.imread(msi_path), cv2.COLOR_BGR2RGB)
    h, w, _ = img_h.shape
    gray_h = cv2.cvtColor(img_h, cv2.COLOR_RGB2GRAY)
    gray_m = cv2.cvtColor(img_m, cv2.COLOR_RGB2GRAY)
    master_mask = np.logical_or((gray_h > 10), (gray_m > 10)).astype(np.float32) # manually combine HSI & MSI masks for viewability
    heatmap = np.zeros((h, w), dtype=np.float32) # store probabilities
    counts = np.zeros((h, w), dtype=np.float32) # store number of times a speicfic pixel is contained in a sliding window
    with torch.no_grad():
        for y in range(0, h - window_size + 1, stride):
            for x in range(0, w - window_size + 1, stride):
                patch_h = img_h[y:y + window_size, x:x + window_size] # localized path
                patch_m = img_m[y:y + window_size, x:x + window_size]
                if mode == "fusion": # get HSI and MSI patches
                    t_h = test_transform(patch_h).unsqueeze(0).to(device)
                    t_m = test_transform(patch_m).unsqueeze(0).to(device)
                    score = model(t_h, t_m).item() # mesh into singular probability
                elif mode == "hsi": # HSI patch only
                    t_h = test_transform(patch_h).unsqueeze(0).to(device)
                    score = model(t_h).item()
                else: # MSI patch only
                    t_m = test_transform(patch_m).unsqueeze(0).to(device)
                    score = model(t_m).item()
                heatmap[y:y+window_size, x:x+window_size] += score
                counts[y:y+window_size, x:x+window_size] += 1
    heatmap = (heatmap / (counts + 1e-8)) * master_mask # average
    heatmap = gaussian_filter(heatmap, sigma=2) * master_mask # smoothen with Gaussian filter for viewability
    if np.max(heatmap) > 0: # normalize
        h_min, h_max = np.min(heatmap[master_mask > 0]), np.max(heatmap[master_mask > 0])
        heatmap = (heatmap - h_min) / (h_max - h_min + 1e-8)
        heatmap = np.clip(heatmap, 0, 1) * master_mask
    return heatmap, master_mask

In [ ]:
# outputs all saliency maps from all implemented models (CNN, SVM, MLP)
def visualize_all_models(idx_val, results_df):
    row = results_df.iloc[idx_val]
    h_p, m_p = row['processed_path_hsi'], row['processed_path_msi'] # load in paths
    h_rgb = cv2.cvtColor(cv2.imread(h_p), cv2.COLOR_BGR2RGB)
    m_rgb = cv2.cvtColor(cv2.imread(m_p), cv2.COLOR_BGR2RGB)
    t_h = test_transform(h_rgb).unsqueeze(0).to(device) # prepare PyTorch tensors 
    t_m = test_transform(m_rgb).unsqueeze(0).to(device)
    bg_h = h_rgb.astype(np.float32) / 255.0 # normalize original (to become background) for heat map overlay
    bg_m = m_rgb.astype(np.float32) / 255.0

    real_id = row['num'] # extract the image number from image name
    fig, axes = plt.subplots(3, 3, figsize=(12, 12), dpi=600) 
    main_title = f"Saliency Maps (Kidney Scan #{real_id}) \nGround Truth: {'Tumor (1)' if row['label']==1 else 'Normal (0)'}"
    fig.suptitle(main_title, fontsize=18)

    # Grad-CAM (CNN)
    axes[0,0].imshow(cnn_gradcam(final_model_hsi, t_h, bg_h, final_model_hsi.conv2))
    axes[0,0].set_title(f"CNN HSI Only (Pred: {row['cnn_hsi']})")
    axes[0,1].imshow(cnn_gradcam(final_model_msi, t_m, bg_m, final_model_msi.conv2))
    axes[0,1].set_title(f"CNN MSI Only (Pred: {row['cnn_msi']})")
    axes[0,2].imshow(fusion_gradcam(final_model_fusion, t_h, t_m, bg_h, final_model_fusion.hsi_bn2))
    axes[0,2].set_title(f"CNN Late Fusion (Pred: {row['cnn_fusion']})")
    axes[0,0].set_ylabel("CNN (Grad-CAM)", fontsize=15)

    # sliding window (MLP)
    mlp_configs = [("MLP HSI Only", "hsi", final_model_mlp_hsi, "mlp_hsi", bg_h),
                   ("MLP MSI Only", "msi", final_model_mlp_msi, "mlp_msi", bg_m),
                   ("MLP Late Fusion", "fusion", final_mlp_fusion_model, "mlp_fusion", bg_h)]
    for i, (display_name, mode, mod, col, bg) in enumerate(mlp_configs):
        hmap, msk = generate_mlp_saliency_unified(h_p, m_p, mod, mode) # saliency map
        hmap_c = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET) # transform normalized probability into jet colormap
        hmap_c = cv2.cvtColor(hmap_c, cv2.COLOR_BGR2RGB)
        hmap_c[msk == 0] = 0 # eliminate backgruond noise for viewability
        axes[1,i].imshow(cv2.addWeighted(bg, 0.6, hmap_c.astype(np.float32)/255, 0.4, 0))
        axes[1,i].set_title(f"{display_name} (Pred: {row[col]})")
    axes[1,0].set_ylabel("MLP (Sliding Window)", fontsize=15)

    # sliding window (SVM)
    svm_configs = [("SVM HSI Only", final_svm_hsi_only, scaler_svm_hsi_only, bg_h, "svm_hsi"),
                   ("SVM MSI Only", final_svm_msi_only, scaler_svm_msi_only, bg_m, "svm_msi"),
                   ("SVM Early Fusion", final_svm_early_fusion, scaler_svm_early_fusion, bg_h, "svm_fusion")]
    for i, (display_name, mod, scal, bg, col) in enumerate(svm_configs):
        internal_name = "hsi only" if "HSI" in display_name else "msi only" if "MSI" in display_name else "early fusion"
        hmap, _, msk = generate_svm_saliency_unified(h_p, m_p, mod, scal, internal_name)
        hmap_c = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        hmap_c = cv2.cvtColor(hmap_c, cv2.COLOR_BGR2RGB)
        hmap_c[msk == 0] = 0 
        axes[2,i].imshow(cv2.addWeighted(bg, 0.6, hmap_c.astype(np.float32)/255, 0.4, 0))
        axes[2,i].set_title(f"{display_name} (Pred: {row[col]})")
    axes[2,0].set_ylabel("SVM (Sliding Window)", fontsize=15)

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
        if not ax.get_ylabel():
            ax.axis('off')
            
    plt.savefig('saliencymap.png', dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
img_size_tuple = (128, 128) # Ensure this matches your window_size

final_model_mlp_hsi = MLP(input_size=img_size_tuple, channels=3).to(device) 
final_model_mlp_msi = MLP(input_size=img_size_tuple, channels=3).to(device)
fusion_input_dim = img_size_tuple[0] * img_size_tuple[1] * 3
final_mlp_fusion_model = LateFusionMLP(hsi_dim=fusion_input_dim, msi_dim=fusion_input_dim).to(device)

final_model_mlp_hsi.load_state_dict(torch.load('final_mlp_hsi_model.pth', map_location=device))
final_model_mlp_msi.load_state_dict(torch.load('final_mlp_msi_model.pth', map_location=device))
final_mlp_fusion_model.load_state_dict(torch.load('final_mlp_fusion_model.pth', map_location=device))

In [ ]:
results_df = build_diagnostic_db(final_test_df) # first time

In [ ]:
results_df.to_csv("heat_map_results_v3.csv", index=False) # calling a saved CSV (convenient)

In [ ]:
results_df = pd.read_csv("heat_map_results_v3.csv") # read from saved CSV
case_scenario = np.where((results_df['label'] == 1) & 
                         (results_df['svm_msi'] == 0) & (results_df['svm_hsi'] == 0) & (results_df['svm_fusion'] == 0))[0]
print(f"Number of matching scenarios: {len(case_scenario)}")
if len(case_scenario) > 0:
    visualize_all_models(case_scenario[26], results_df)